## PROJECT: Sentiment Analysis for Alexa Reviews in Amazon
- Reading the data - Alexa Reviews are downloaded from Kaggle
- Text Preprocessing
- One hot encoding
- BoW
- TF- IDF
- ML

## Reading the data

In [47]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("sid321axn/amazon-alexa-reviews")

# print("Path to dataset files:", path)

#Reading the data
import pandas as pd
data = pd.read_csv(r"D:\GEN_AI_BOOTCAMP\Live_Classes1\Assignments\Data\amazon_alexa.tsv", sep='\t')
data.head()
#distinct values in the feedback column
#cross tab between rating and feedback

# data.to_excel(r"D:\GEN_AI_BOOTCAMP\Live_Classes1\Assignments\Data\alexa_reviews.xlsx", index=False)    #Output for Sunny
pd.crosstab(data['rating'], data['feedback'])

feedback,0,1
rating,,
1,161,0
2,96,0
3,0,152
4,0,455
5,0,2286


#### The dataset shows sentiment 1 for rating above 2 and 1,2 as negative sentiment

In [48]:
#print the column names
print(data.columns)

Index(['rating', 'date', 'variation', 'verified_reviews', 'feedback'], dtype='str')


In [49]:
#Converting the reviews in list of sentence
data = data[data["verified_reviews"].apply(lambda x: isinstance(x, str))] # removing records with non-string reviews
data = data[data["verified_reviews"] != " "] # removing records with blank reviews
reviews = data["verified_reviews"].tolist()
print(pd.crosstab(data['rating'], data['feedback']))
print(reviews[:5])

feedback    0     1
rating             
1         146     0
2          91     0
3           0   140
4           0   447
5           0  2246
['Love my Echo!', 'Loved it!', 'Sometimes while playing a game, you can answer a question correctly but Alexa says you got it wrong and answers the same as you.  I like being able to turn lights on and off while away from home.', 'I have had a lot of fun with this thing. My 4 yr old learns about dinosaurs, i control the lights and play games like categories. Has nice sound when playing music as well.', 'Music']


###### 3 non strings records removed

## Text Preprocessing

In [50]:
#Text preprocessing including lowercasing, removing punctuation and stop words,stopwords,lametisation
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# nltk.download("stopwords")
# nltk.download("wordnet")
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(lemmatized_tokens)
preprocessed_reviews = [preprocess_text(review) for review in reviews]

#Removing Punctuation
import string
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))
reviews_cleaned = [remove_punctuation(review) for review in preprocessed_reviews]


#remove hyphens
def remove_hyphens(text):
    return text.replace("-", " ")
reviews_cleaned = [remove_hyphens(review) for review in reviews_cleaned]
reviews_cleaned[:5]

['love echo',
 'loved',
 'sometimes playing game answer question correctly alexa say got wrong answer like able turn light away home',
 'lot fun thing 4 yr old learns dinosaur control light play game like category nice sound playing music well',
 'music']

In [51]:
#Need to merge the cleaned reviews with the main dataframe
data["cleaned_review"] = reviews_cleaned
data=data[data["cleaned_review"] != " "]
data=data[data["cleaned_review"] != ""]  # removing records with blank cleaned reviews
len(data)


3063

## Tokenisation

In [59]:
#Tokenisation of reviews_cleaned
reviews_cleaned = data["cleaned_review"].tolist()
tokenized_reviews = [review.split() for review in reviews_cleaned]
tokenized_reviews [:5]


[['love', 'echo'],
 ['loved'],
 ['sometimes',
  'playing',
  'game',
  'answer',
  'question',
  'correctly',
  'alexa',
  'say',
  'got',
  'wrong',
  'answer',
  'like',
  'able',
  'turn',
  'light',
  'away',
  'home'],
 ['lot',
  'fun',
  'thing',
  '4',
  'yr',
  'old',
  'learns',
  'dinosaur',
  'control',
  'light',
  'play',
  'game',
  'like',
  'category',
  'nice',
  'sound',
  'playing',
  'music',
  'well'],
 ['music']]

In [60]:
#Specific format
all_words = [[word] for review in tokenized_reviews for word in review]
all_words[:5]

[['love'], ['echo'], ['loved'], ['sometimes'], ['playing']]

## One hot encoding

In [61]:
#fit one hot encoding
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder()
encoder.fit(all_words)
print("Vocabulary:", encoder.categories_[0])
encoder.categories_[0].shape

Vocabulary: ['072318' '1' '10' ... 'zzzzzzz' 'í' 'útil']


(4021,)

In [56]:
a = 0
for idx, sentence in enumerate(tokenized_reviews):
    arr = np.array(sentence)
   
    if arr.ndim == 1:
        a=a+1   
    
print(a)

3070


In [62]:
#One hot encoding for reviews cleaned
import numpy as np
for sentence in tokenized_reviews:
    # sentence=np.array(sentence).reshape(-1, 1)
    encoded_sentence = encoder.transform([[word] for word in sentence])
    print("\nSentence:", sentence)
    print(encoded_sentence)
    print("Dimension:", encoded_sentence.shape)


Sentence: ['love', 'echo']
  (0, 2154)	1.0
  (1, 1190)	1.0
Dimension: (2, 4021)

Sentence: ['loved']
  (0, 2155)	1.0
Dimension: (1, 4021)

Sentence: ['sometimes', 'playing', 'game', 'answer', 'question', 'correctly', 'alexa', 'say', 'got', 'wrong', 'answer', 'like', 'able', 'turn', 'light', 'away', 'home']
  (0, 3275)	1.0
  (1, 2633)	1.0
  (2, 1526)	1.0
  (3, 325)	1.0
  (4, 2793)	1.0
  (5, 893)	1.0
  (6, 262)	1.0
  (7, 3048)	1.0
  (8, 1587)	1.0
  (9, 3980)	1.0
  (10, 325)	1.0
  (11, 2087)	1.0
  (12, 151)	1.0
  (13, 3693)	1.0
  (14, 2077)	1.0
  (15, 433)	1.0
  (16, 1733)	1.0
Dimension: (17, 4021)

Sentence: ['lot', 'fun', 'thing', '4', 'yr', 'old', 'learns', 'dinosaur', 'control', 'light', 'play', 'game', 'like', 'category', 'nice', 'sound', 'playing', 'music', 'well']
  (0, 2148)	1.0
  (1, 1505)	1.0
  (2, 3571)	1.0
  (3, 109)	1.0
  (4, 4010)	1.0
  (5, 2445)	1.0
  (6, 2056)	1.0
  (7, 1076)	1.0
  (8, 868)	1.0
  (9, 2077)	1.0
  (10, 2630)	1.0
  (11, 1526)	1.0
  (12, 2087)	1.0
  (13, 665)

## BOW [Just for practice, doing again in ML Pipeline]

In [63]:
#Count vectorisation for reviews_cleaned
from sklearn.feature_extraction.text import CountVectorizer
cnt_vect = CountVectorizer()
cnt_vect.fit(reviews_cleaned)
encoded_reviews_cnt_vect = cnt_vect.fit_transform(reviews_cleaned)
print("dimension:", cnt_vect.get_feature_names_out().shape[0])
print("Vocabulary:", cnt_vect.get_feature_names_out())
print(encoded_reviews_cnt_vect.toarray())

#Merging the BOW to the main data for further machine learning model building

encoded_reviews_df_bow = pd.DataFrame(encoded_reviews_cnt_vect.toarray(), columns=cnt_vect.get_feature_names_out())
final_df_cnt_bow= pd.concat([data, encoded_reviews_df_bow], axis=1)    
final_df_cnt_bow.head()


dimension: 4000
Vocabulary: ['072318' '10' '100' ... 'zzzz' 'zzzzzzz' 'útil']
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


,rating,date,variation,verified_reviews,feedback,cleaned_review,072318,10,100,1000,...,youve,yr,yup,zero,zigbee,zonkedout,zwave,zzzz,zzzzzzz,útil
0,5.0,31-Jul-18,Charcoal Fabric,Love my Echo!,1.0,love echo,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5.0,31-Jul-18,Charcoal Fabric,Loved it!,1.0,loved,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer...",1.0,sometimes playing game answer question correct...,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5.0,31-Jul-18,Charcoal Fabric,I have had a lot of fun with this thing. My 4 ...,1.0,lot fun thing 4 yr old learns dinosaur control...,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.0,31-Jul-18,Charcoal Fabric,Music,1.0,music,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## TF-IDF [Just for practice - doing again in ML Pipeline]

In [64]:
#TF IDF with reviews_cleaned
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(reviews_cleaned)
encoded_reviews_tfidf_vect = tfidf_vect.fit_transform(reviews_cleaned)
print("dimension:", tfidf_vect.get_feature_names_out().shape[0])
print("Vocabulary:", tfidf_vect.get_feature_names_out())
print(encoded_reviews_tfidf_vect.toarray())

#Merging the BOW to the main data for further machine learning model building

encoded_reviews_df_tfidf = pd.DataFrame(encoded_reviews_tfidf_vect.toarray(), columns=tfidf_vect.get_feature_names_out())
final_df_cnt_tfidf= pd.concat([data,encoded_reviews_df_tfidf], axis=1)    
final_df_cnt_tfidf.head()



dimension: 4000
Vocabulary: ['072318' '10' '100' ... 'zzzz' 'zzzzzzz' 'útil']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


,rating,date,variation,verified_reviews,feedback,cleaned_review,072318,10,100,1000,...,youve,yr,yup,zero,zigbee,zonkedout,zwave,zzzz,zzzzzzz,útil
0,5.0,31-Jul-18,Charcoal Fabric,Love my Echo!,1.0,love echo,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5.0,31-Jul-18,Charcoal Fabric,Loved it!,1.0,loved,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer...",1.0,sometimes playing game answer question correct...,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5.0,31-Jul-18,Charcoal Fabric,I have had a lot of fun with this thing. My 4 ...,1.0,lot fun thing 4 yr old learns dinosaur control...,0.0,0.0,0.0,0.0,...,0.0,0.327934,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5.0,31-Jul-18,Charcoal Fabric,Music,1.0,music,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## ML

In [ ]:
#lets fit a logistic regression model on the count vectorised data
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

log_reg = LogisticRegression()

#test train split for the data
#60% train, 20% test, 20% validation
X_train, X_temp, y_train, y_temp = train_test_split(data['cleaned_review'], data['feedback'], test_size=0.4, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42) 


#Saving the validation data for later use
data_val= pd.DataFrame({'review': X_val, 'feedback': y_val})


In [ ]:
# Name X_val a column name


,review,feedback
1645,easy use answer questionshands free whats like...,1
581,bought mother love itshe love guard dog featur...,1
2334,prime day purchase im still learning way aroun...,1
3078,love added upstairs bedroom,1
552,second dot work great refurbishedthought new,1


## ML with BOW

In [83]:
#Vectorising with BOW

from sklearn.feature_extraction.text import CountVectorizer
bow= CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

#ML Training
log_reg.fit(X_train_bow, y_train)

probs=log_reg.predict_proba(X_test_bow)[:,1]
threshold=0.6
y_pred = (probs >= threshold).astype(int)

print(classification_report(y_test, y_pred))
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)


              precision    recall  f1-score   support

           0       0.68      0.39      0.50        38
           1       0.96      0.99      0.97       575

    accuracy                           0.95       613
   macro avg       0.82      0.69      0.74       613
weighted avg       0.94      0.95      0.94       613

Confusion Matrix:
 [[ 15  23]
 [  7 568]]


## TF IDF

In [74]:

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()


X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

#ML Training
log_reg.fit(X_train_tfidf, y_train)

probs=log_reg.predict_proba(X_test_tfidf)[:,1]
threshold=0.7
y_pred = (probs >= threshold).astype(int)

print(classification_report(y_test, y_pred))
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)



              precision    recall  f1-score   support

           0       0.75      0.08      0.14        38
           1       0.94      1.00      0.97       575

    accuracy                           0.94       613
   macro avg       0.85      0.54      0.56       613
weighted avg       0.93      0.94      0.92       613

Confusion Matrix:
 [[  3  35]
 [  1 574]]


## Comparison with completely new data


In [81]:
# data_val = pd.read_csv(r"D:\GEN_AI_BOOTCAMP\Live_Classes1\Assignments\Data\new_amazon_reviews.csv")
new_review_cleaned = data_val["review"].apply(preprocess_text)
new_review_cleaned = new_review_cleaned.apply(remove_punctuation)
new_review_cleaned = new_review_cleaned.apply(remove_hyphens)
Threshold_pred=0.6

#Vectorising the new review
new_review_bow= bow.transform(new_review_cleaned)   #USING TRANSFORM ONLY AS VECTORISOR IS ALREADY FITTED ON THE TRAINING DATA
new_review_tfidf = tfidf.transform(new_review_cleaned)


pred_prob_bow = log_reg.predict_proba(new_review_bow)
pred_bow= (pred_prob_bow[:, 1] >= Threshold_pred).astype(int)
pred_prob_tfidf = log_reg.predict_proba(new_review_tfidf)
pred_tfidf= (pred_prob_tfidf[:, 1] >= Threshold_pred).astype(int)

confusion_matrix_bow = confusion_matrix(data_val['feedback'], pred_bow)
confusion_matrix_tfidf = confusion_matrix(data_val['feedback'], pred_tfidf)

accuracy_bow = (confusion_matrix_bow[0, 0] + confusion_matrix_bow[1, 1]) / confusion_matrix_bow.sum()
accuracy_tfidf = (confusion_matrix_tfidf[0, 0] + confusion_matrix_tfidf[1, 1]) / confusion_matrix_tfidf.sum()

precision_bow = confusion_matrix_bow[1, 1] / (confusion_matrix_bow[0, 1] + confusion_matrix_bow[1, 1])
precision_tfidf = confusion_matrix_tfidf[1, 1] / (confusion_matrix_tfidf[0, 1] + confusion_matrix_tfidf[1, 1])

recall_bow = confusion_matrix_bow[1, 1] / (confusion_matrix_bow[1, 0] + confusion_matrix_bow[1, 1])
recall_tfidf = confusion_matrix_tfidf[1, 1] / (confusion_matrix_tfidf[1, 0] + confusion_matrix_tfidf[1, 1])

f1_score_bow = 2 * (precision_bow * recall_bow) / (precision_bow + recall_bow)
f1_score_tfidf = 2 * (precision_tfidf * recall_tfidf) / (precision_tfidf + recall_tfidf)

print("Confusion Matrix for BOW:\n", confusion_matrix_bow)
print("Confusion Matrix for TF-IDF:\n", confusion_matrix_tfidf)

print(f"Accuracy with BOW: {accuracy_bow:.2f}")
print(f"Accuracy with TF-IDF: {accuracy_tfidf:.2f}")
print(f"Precision with BOW: {precision_bow:.2f}")
print(f"Precision with TF-IDF: {precision_tfidf:.2f}")
print(f"Recall with BOW: {recall_bow:.2f}")
print(f"Recall with TF-IDF: {recall_tfidf:.2f}")
print(f"F1 Score with BOW: {f1_score_bow:.2f}")
print(f"F1 Score with TF-IDF: {f1_score_tfidf:.2f}")

#Merge the predictions with the main data
data_val["predicted_label_bow"] = pred_bow
data_val["predicted_label_tfidf"] = pred_tfidf
data_val["predicted_prob_bow"] = pred_prob_bow[:, 1]
data_val["predicted_prob_tfidf"] = pred_prob_tfidf[:, 1]
data_val.head()





    

Confusion Matrix for BOW:
 [[ 22  31]
 [ 15 545]]
Confusion Matrix for TF-IDF:
 [[  1  52]
 [  0 560]]
Accuracy with BOW: 0.92
Accuracy with TF-IDF: 0.92
Precision with BOW: 0.95
Precision with TF-IDF: 0.92
Recall with BOW: 0.97
Recall with TF-IDF: 1.00
F1 Score with BOW: 0.96
F1 Score with TF-IDF: 0.96


,review,feedback,predicted_label_bow,predicted_label_tfidf,predicted_prob_bow,predicted_prob_tfidf
1645,easy use answer questionshands free whats like...,1,1,1,0.978474,0.906733
581,bought mother love itshe love guard dog featur...,1,1,1,0.999981,0.968189
2334,prime day purchase im still learning way aroun...,1,1,1,0.997035,0.957920
3078,love added upstairs bedroom,1,1,1,0.998222,0.963610
552,second dot work great refurbishedthought new,1,1,1,0.992298,0.954631


In [82]:
# EXPORT TO EXCEL
data_val.to_excel(r"D:\GEN_AI_BOOTCAMP\Live_Classes1\Assignments\Data\alexa_reviews_with_predictions.xlsx", index=False)
